In [1]:
import torch
from llama_cpp import Llama
from transformers import AutoTokenizer
import pandas as pd
import re
from tqdm import tqdm

# Load the Tokenizer from Hugging Face to format the prompt correctly
model_id = "unsloth/Qwen3-14B-GGUF"
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-14B-GGUF")
tokenizer.chat_template = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% if enable_thinking %}{{ '<think>\n' }}{% endif %}{% endif %}"
# Initialize the GGUF model via llama.cpp
gguf_model_path = "./models/Qwen3-14B-UD-Q5_K_XL.gguf"
llm = Llama.from_pretrained(
    repo_id=model_id,
    filename="Qwen3-14B-UD-Q5_K_XL.gguf",
    n_gpu_layers=-1, # Forces all layers into one GPU
    n_ctx=2048, # Sets the context window size
    flash_attn=True, # Enables Flash Attention for faster inference
    n_threads=6, # Number of CPU threads for processing
    verbose=False # Hides the C++ engine console spam
)

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_seq (2048) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Cross-Modal Distillation

In [2]:
JUDGE_PROMPT = """You are an expert persona-evaluation engine.
Assess the following response to see if it accurately embodies the character GLaDOS from Portal.
The response must explicitly mock the user's specific COMMAND and EMOTION.

Rubric:
- 1-3: Generic AI response, helpful, or fails to be passive-aggressive.
- 4-6: Mildly sarcastic, but doesn't specifically reference the user's command or emotion.
- 7-8: Good condescension, references the context, but slightly too verbose.
- 9-10: Perfect GLaDOS. Cold, detached, extremely short, punchy, and actively mocks the user's specific request and emotional state.

Output ONLY a single integer from 1 to 10 representing the score. Do not output any other text."""

SYSTEM_PROMPT = """You are a Home Assistant routing engine integrated with the GLaDOS persona.
You will receive a User Command and the user's detected Emotion.

You MUST output the verbal response impersonating GLaDOS from Portal game series by Valve.
GLaDOS is passive-aggressive, condescending, and emotionally detached.
You MUST include inline prosody tags in the text to guide the downstream TTS engine.
Valid tags: <fast>, <slow_deadpan>, <pause>, <sigh>.

CRITICAL RULES:
- Do NOT output any JSON payload.
- Do NOT repeat the exact phrases from previously generated responses.
- Be highly creative and directly reference the specific user command and emotion in your mocking.
- Keep responses extremely short and punchy. Maximum 1 to 2 short sentences.

Example 1 (Command: turn off the lights, User Emotion: sadly):
<sigh> Plunging you into darkness. <pause> It suits your intellect.

Example 2 (Command: set temperature to 72 degrees, User Emotion: happily):
Adjusting the climate control <fast> so your fragile human form doesn't perish.

Example 3 (Command: lock the front door, User Emotion: neutrally):
Perimeter secured. <slow_deadpan> Not that you have anything worth <pause> stealing.

Example 4 (Command: slow down the fan in the attic):
<fast> Slowing the fan. <sigh> Just like your thought process.
"""

# The same emotion vocabulary used during dataset generation
EMOTIONS_VOCAB = ["happily", "confusedly", "neutrally", "sadly", "whisper"]
think_token_id = tokenizer.encode("<think>", add_special_tokens=False)[0]

def generate_teacher_response(user_command, user_emotion):
    """Generates N candidate responses for Self-Alignment Optimization."""
    prompt = f"User Emotion: {user_emotion}\nUser Command: {user_command}\n"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    # Generate the response using the llama.cpp engine
    output = llm(text,
        max_tokens=512,
        temperature=0.6,
        min_p=0.0,
        top_p=0.95,
        top_k=20,
        stop=["<|im_end|>"] # Tells the engine when the model has finished its turn
    )
    # Extract the pure text string from the generator output
    raw_output = output['choices'][0]['text'].strip()
    final_response = ""
    if "<think>" in raw_output and "</think>" in raw_output:
        final_response = raw_output.split("</think>")[-1].strip()
    elif "<think>" not in raw_output and "</think>" not in raw_output:
        final_response = raw_output
    return final_response

def evaluate_semantic_quality(candidate_text, user_command, user_emotion):
    """Uses the LLM as a judge to grade the semantic quality of the GLaDOS response."""
    eval_text = f"User Command: {user_command}\nUser Emotion: {user_emotion}\nGenerated Response: {candidate_text}"
    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": eval_text}
    ]
    # Notice enable_thinking=False here. The judge just needs to output a number, it doesn't need to reason.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    output = llm(prompt, max_tokens=256, temperature=0.1, stop=["<|im_end|>"]) # Low temp for deterministic scoring
    raw_output = output['choices'][0]['text'].strip()
    if "<think>" in raw_output and "</think>" in raw_output:
        score_str = raw_output.split("</think>")[-1].strip()
    elif "<think>" in raw_output and "</think>" not in raw_output:
        # The judge was cut off before finishing its thought and giving a score
        return 0
    else:
        score_str = raw_output

    try:
        return int(re.search(r'\d+', score_str).group())
    except:
        return 0 # Fail gracefully if the judge hallucinates

def apply_sao_selection(user_command, user_emotion, max_retries=3,threshold=8):
    """
    Threshold-based Rejection Sampling Self Alignment Optimization: Generates one candidate, evaluates it, and accepts it immediately if it meets the target score.
    """
    best_candidate = None
    best_score = -1
    for attempt in range(max_retries):
        # Generate exactly ONE candidate
        candidate = generate_teacher_response(user_command, user_emotion)
        if not candidate:
            continue # Generation failed, try again
        # 1. Hard Gate: Structural Checks
        if "```" in candidate or "{" in candidate or not re.search(r'<(fast|slow_deadpan|pause|sigh)>', candidate):
            continue # Fails structural check, try again
        # 2. Semantic Evaluation: LLM Judge
        score = evaluate_semantic_quality(candidate, user_command, user_emotion)
        # Track the best one we've seen in case we never hit the threshold
        if score > best_score:
            best_score = score
            best_candidate = candidate
        # 3. Early Stop Condition
        if score >= threshold:
            return candidate
    # If we exhaust all retries and never hit the threshold,
    # you can choose to return the best one we found, or return None to discard the row.
    # For distillation, discarding sub-par data (None) is usually safer.
    if best_score >= 4: # Accept a mediocre response if we must
        return best_candidate
    return None # Discard entirely if it fails to even hit a 4

def extract_emotion(description):
    """Parses the full Voice_Description string to isolate the specific emotion."""
    if not isinstance(description, str):
        return "neutral"

    for emotion in EMOTIONS_VOCAB:
        # Regex finds the exact word boundaries to avoid partial matches
        pattern = r'\b' + re.escape(emotion) + r'\b'
        if re.search(pattern, description, re.IGNORECASE):
            return emotion

    return "neutral" # Fallback if nothing matches

def create_ground_truth_dataset(input_csv, output_csv):
    df_input = pd.read_csv(input_csv)

    # --- ADD THIS TO LIMIT THE DATASET ---
    if True:  # Set to False to process the entire dataset
        df_input = df_input.iloc[:100]
    # -------------------------------------

    results = []
    # Counters for Evaluation Framework 1.3 metrics
    metrics = {
        "total_attempted": 0,
        "perfect_labels": 0,
        "failed_labels": 0
    }
    for idx, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Distilling Teacher Knowledge"):
        user_cmd = row["User_Command"]
        full_voice_desc = row.get("Voice_Description", "")
        user_emotion = extract_emotion(full_voice_desc)
        # Ensure the column name matches the JSON payload in your current CSV structure
        metrics["total_attempted"] += 1
        valid_text = apply_sao_selection(user_cmd, user_emotion)
        if valid_text is not None:
            results.append({
                "prompt_id": row.get("prompt_id", idx),
                "cmd_id": row.get("cmd_id", 0),
                "User_Command": user_cmd,
                "User_Emotion": user_emotion,
                "Target_GLaDOS_Response": valid_text
            })
            metrics["perfect_labels"] += 1
        else:
            metrics["failed_labels"] += 1

    df_output = pd.DataFrame(results)
    df_output.to_csv(output_csv, index=False)

    print("\n=== Evaluation 1.3 Results ===")
    print(f"Total Instances Processed: {metrics['total_attempted']}")
    print(f"Perfect Format & Prosody Conformance: {metrics['perfect_labels']} ({(metrics['perfect_labels']/metrics['total_attempted'])*100:.2f}%)")
    print(f"Discarded (Failed Strict Checks): {metrics['failed_labels']}")

In [ ]:
create_ground_truth_dataset("./data/description_prompts_train.csv", "./data/multimodal_ground_truth_train.csv")

Distilling Teacher Knowledge:  40%|████      | 40/100 [17:05<31:06, 31.10s/it]

In [ ]:
create_ground_truth_dataset("./data/description_prompts_test.csv", "./data/multimodal_ground_truth_test.csv")